### Imports and settings

In [ ]:
# Imports
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from tqdm.auto import tqdm
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

VOTE_THRESHOLD = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device}")
if device.type == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### Data and model folders (local or Colab)

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/vizwiz")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/vizwiz_checkpoints")
    BIS_MODEL_DIR = Path("/content/drive/MyDrive/vizwiz_bis_model")
    IN_COLAB = True
except ImportError:
    DATA = Path(os.getenv("DATA_PATH"))
    CHECKPOINT_DIR = Path.cwd() / "checkpoints"
    BIS_MODEL_DIR = Path.cwd() / "bis_model"
    IN_COLAB = False

ANNOTATIONS = DATA / "annotations"
IMG = DATA / "images"

print("Data folder:", DATA, "| exists:", DATA.exists())
print("History folder:", CHECKPOINT_DIR, "| exists:", CHECKPOINT_DIR.exists())
print("Model folder:", BIS_MODEL_DIR, "| exists:", BIS_MODEL_DIR.exists())

### Dataset class (same as the main notebook)

In [ ]:
LABELS = ["BLR","BRT","DRK","FRM","OBS","OTH","NON","ROT"]

class VizWizQuality (Dataset):

    def __init__ (self, json_path, img_dir, transform):
        self.items = json.load(open(json_path))
        self.img_dir = Path(img_dir)
        self.transform =  transform

    def __len__ (self): return len(self.items)

    def __getitem__ (self, i):
        it = self.items[i]
        img = Image.open (self.img_dir / it["image"]).convert("RGB")
        x = self.transform(img)

        # checking if there is no data label, then return the filename
        if "flaws" not in it:
            return x, it["image"]

        y = [it["flaws"][k] / 5 for k in LABELS] + [it["unrecognizable"] / 5]
        return x, torch.tensor (y, dtype = torch.float32)


### Validation set at 224 x 224

In [ ]:
# same ImageNet normalization the models were trained with
model_weights = ResNet18_Weights.DEFAULT
ORIGINAL_MEAN = model_weights.transforms().mean
ORIGINAL_STD = model_weights.transforms().std

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(ORIGINAL_MEAN, ORIGINAL_STD)
])

val_dataset = VizWizQuality(ANNOTATIONS / "val.json", IMG / "val", eval_transform)
loader_args = dict(batch_size=64, num_workers=0, pin_memory=(device.type == "cuda"))

print("Val dataset size:", len(val_dataset))

### Evaluation functions (same as the main notebook)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def evaluate (model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    correct = torch.zeros(len(LABELS) + 1)
    always_no = torch.zeros(len(LABELS) + 1)

    with torch.no_grad():
        for x, y in tqdm(loader, desc = "Evaluating"):
            x, y = x.to(device), y.to(device)
            logits = model (x)
            total_loss += criterion(logits, y).item() * x.size(0)  # returns the average loss of this batch

            preds = torch.sigmoid(logits) > 0.5  # threshold for if a flaw exist
            truth = torch.round(y * 5) >= VOTE_THRESHOLD  # at least 2 of 5 annotators chose the flaw
            correct += (preds == truth).sum(dim=0).cpu()
            always_no += (~truth).sum(dim=0).cpu()
            n += x.size(0)

        return total_loss / n, correct / n, always_no / n

In [ ]:
def evaluate_detailed(model, loader):
    model.eval()

    labels = LABELS + ["UNREC"]
    tp = torch.zeros(len(labels)) #Each of these four is a true/false positivity/negativity. It seperates results for balanced accuracy
    tn = torch.zeros(len(labels))
    fp = torch.zeros(len(labels))
    fn = torch.zeros(len(labels))

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x = x.to(device)

            logits = model(x)
            preds = (torch.sigmoid(logits) > 0.5).cpu() #Comparing results to actual values
            truth = torch.round(y * 5) >= VOTE_THRESHOLD  # at least 2 of 5 annotators

            tp += (preds & truth).sum(dim=0) #Counting each result type
            tn += (~preds & ~truth).sum(dim=0)
            fp += (preds & ~truth).sum(dim=0)
            fn += (~preds & truth).sum(dim=0)

    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total
    baseline = (tn + fp) / total

    precision = tp / (tp + fp).clamp(min=1)
    recall = tp / (tp + fn).clamp(min=1)
    specificity = tn / (tn + fp).clamp(min=1)

    balanced_accuracy = (recall + specificity) / 2
    balanced_accuracy = torch.where(
        ((tp + fn) > 0) & ((tn + fp) > 0),
        balanced_accuracy,
        torch.full_like(balanced_accuracy, float("nan"))
    )

    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)

    return pd.DataFrame({
        "Label": labels,
        "Accuracy": accuracy.numpy(),
        "Baseline": baseline.numpy(),
        "Improvement (pp)": ((accuracy - baseline) * 100).numpy(),
        "Balanced Accuracy": balanced_accuracy.numpy(),
        "Precision": precision.numpy(),
        "Recall": recall.numpy(),
        "F1": f1.numpy()
    })

## Choose a run

Set `RUN_NAME` to one of the names in `RUNS`. The learning curve, per-label and detailed results below all show that run. `RUN_NAME` is also used in the names of the saved plot and CSV.

In [ ]:
# name: (model, input size, model file in BIS_MODEL_DIR, history file in CHECKPOINT_DIR)
RUNS = {
    # retrained with the 2-of-5 vote rule
    "112426-0053_resnet18_224": (models.resnet18, 224, "112426-0053_resnet18_224_bis.pt", "112426-0053_resnet18_224_history.csv"),
    "112426-0130_resnet18_448": (models.resnet18, 448, "112426-0130_resnet18_448_bis.pt", "112426-0130_resnet18_448_history.csv"),
    "112426-0255_cnxt_224":     (models.convnext_tiny, 224, "112426-0255_cnxt_224_bis.pt", "112426-0255_cnxt_224_history.csv"),
    "112426-0348_cnxb_224":     (models.convnext_base, 224, "112426-0348_cnxb_224_bis.pt", "112426-0348_cnxb_224_history.csv"),
    # earlier reference runs, kept under their original names
    "v2":            (models.resnet18, 224, "resnet18_bis_v2.pt", "history_v2.csv"),
    "448":           (models.resnet18, 448, "resnet18_bis_448.pt", "history_448.csv"),
    "ConvNeXt_Tiny": (models.convnext_tiny, 224, "111926-0046_cnxt_bis.pt", "111926-0046_history_cnxt.csv"),
    "ConvNeXt_Base": (models.convnext_base, 224, "111926-1017_cnxb_bis.pt", "111926-1017_history_cnxb.csv"),
}

RUN_NAME = "112426-0255_cnxt_224"

build, RUN_IMG_SIZE, model_file, history_file = RUNS[RUN_NAME]
RUN_MODEL_PATH = BIS_MODEL_DIR / model_file
RUN_HISTORY_PATH = CHECKPOINT_DIR / history_file
history = pd.read_csv(RUN_HISTORY_PATH).to_dict("records")

# validation images at this run's input size, same normalization
run_transform = transforms.Compose([
    transforms.Resize((RUN_IMG_SIZE, RUN_IMG_SIZE)), *eval_transform.transforms[1:]
])
run_val_dataset = VizWizQuality(ANNOTATIONS / "val.json", IMG / "val", run_transform)
RUN_VAL_LOADER = DataLoader(run_val_dataset, shuffle=False, **dict(loader_args, batch_size=32))

# same architecture as in training, with the 9-output head (8 flaws + unrecognizable)
model = build(weights=None)
if build is models.resnet18:
    model.fc = nn.Linear(model.fc.in_features, len(LABELS) + 1)
else:
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(LABELS) + 1)
model = model.to(device)

print(f"run: {RUN_NAME} | {build.__name__} at {RUN_IMG_SIZE}px | {len(history)} epochs")
print("model:", RUN_MODEL_PATH.name, "| history:", RUN_HISTORY_PATH.name)

### Plotting the learning curve

In [ ]:
hist = pd.DataFrame(history).rename(columns={"mean_val_loss": "mean_val_acc"})
print(hist.columns.tolist())
best = hist.loc[hist["val_loss"].idxmin()]     

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
ax1.plot(hist["epoch"], hist["val_loss"],   marker="o", label="val")
ax1.axhline(0.693, linestyle="--", color="gray", label="guessing (0.693)")
ax1.axvline(best["epoch"], linestyle=":", color="green",
            label=f"best epoch ({int(best['epoch'])})")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("BCE loss")
ax1.set_xticks(hist["epoch"])
ax1.legend()

val_truth = torch.tensor([
    [it["flaws"][k] for k in LABELS] + [it["unrecognizable"]]
    for it in val_dataset.items
]) >= VOTE_THRESHOLD
baseline_acc = (~val_truth).float().mean(dim=0)
print("always-'no' accuracy:", baseline_acc.mean().item())

ax2.plot(hist["epoch"], hist["mean_val_acc"], marker="o", color="tab:orange", label="model")
ax2.axhline(baseline_acc.mean().item(), linestyle="--", color="gray", label="always 'no'")
ax2.set_title("Validation accuracy (mean over 9 labels)")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_xticks(hist["epoch"])
ax2.legend()

fig.suptitle(f"run: {RUN_NAME}")
plt.tight_layout()
plot_path = CHECKPOINT_DIR / f"learning_curves_{RUN_NAME}.png"
if plot_path.exists():
    print(f"{plot_path.name} already exists, not overwriting it")
else:
    plt.savefig(plot_path, dpi=150)
plt.show()

### Per-label results of the best model

In [ ]:
model.load_state_dict(torch.load(RUN_MODEL_PATH))
val_loss, val_acc, _ = evaluate(model, RUN_VAL_LOADER)

print(f"run {RUN_NAME}, best model val loss: {val_loss:.4f}")
for name, a, b in zip(LABELS + ["UNREC"], val_acc, baseline_acc):
    print(f"{name:6} model acc: {a:.3f}   always-'no' acc: {b:.3f}")

### Displaying detailed results

In [ ]:
#Load saved model
from pathlib import Path

# the run chosen above
MODEL_PATH = RUN_MODEL_PATH

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)
)

model.eval()

print("Trained model loaded successfully.")

In [ ]:
#Print results
results = evaluate_detailed(model, RUN_VAL_LOADER)

display_results = results[results["Label"] != "OTH"] #Exclude the "Other" catergory
display(display_results.style.format({
    "Accuracy": "{:.1%}",
    "Baseline": "{:.1%}",
    "Improvement (pp)": "{:+.1f}",
    "Balanced Accuracy": "{:.1%}",
    "Precision": "{:.1%}",
    "Recall": "{:.1%}",
    "F1": "{:.1%}"
}))

In [ ]:
#Save results
RESULTS_DIR = Path.cwd() / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / f"per_label_metrics_{RUN_NAME}_vote{VOTE_THRESHOLD}.csv"
if results_path.exists():
    print(f"{results_path.name} already exists, not overwriting it")
else:
    display_results.to_csv(results_path, index=False)
    print("Results saved to", results_path.name)

## Compare trained models

Edit the `runs` list to choose which saved weights in `BIS_MODEL_DIR` are compared.

In [ ]:
def compare_models():
    runs = [
        ("ResNet18 (224)", models.resnet18, "resnet18_bis_v2.pt"),
        ("ResNet18 (448)", models.resnet18, "resnet18_bis_448.pt"),
        ("ConvNeXt Tiny", models.convnext_tiny, "111926-0046_cnxt_bis.pt"),
        ("ConvNeXt Base", models.convnext_base, "111926-1017_cnxb_bis.pt"),
    ]

    # Keep the validation images and normalization the same at both resolutions.
    transform_448 = transforms.Compose([
        transforms.Resize((448, 448)), *eval_transform.transforms[1:]
    ])
    val_448 = VizWizQuality(ANNOTATIONS / "val.json", IMG / "val", transform_448)
    comparison_loader_args = dict(loader_args, batch_size=16)
    loaders = {
        "ResNet18 (448)": DataLoader(val_448, shuffle=False, **comparison_loader_args),
    }
    standard_loader = DataLoader(val_dataset, shuffle=False, **comparison_loader_args)

    tables = []
    for name, build, filename in runs:
        print(f"Evaluating {name}")
        candidate = build(weights=None)
        if build is models.resnet18:
            candidate.fc = nn.Linear(candidate.fc.in_features, len(LABELS) + 1)
        else:
            candidate.classifier[2] = nn.Linear(candidate.classifier[2].in_features, len(LABELS) + 1)
        candidate.load_state_dict(torch.load(BIS_MODEL_DIR / filename, map_location="cpu", weights_only=True))
        metrics = evaluate_detailed(candidate.to(device), loaders.get(name, standard_loader))
        del candidate
        metrics.insert(0, "Model", name)
        tables.append(metrics)

    per_label = pd.concat(tables, ignore_index=True)
    summary = (
        per_label[per_label["Label"] != "OTH"]
        .groupby("Model", as_index=False)[["F1", "Balanced Accuracy"]].mean()
        .rename(columns={"F1": "Macro F1", "Balanced Accuracy": "Mean Balanced Accuracy"})
        .sort_values("Macro F1", ascending=False)
    )
    output = Path("results")
    output.mkdir(exist_ok=True)
    summary.to_csv(output / f"model_comparison_vote{VOTE_THRESHOLD}.csv", index=False)
    per_label.to_csv(output / f"model_comparison_per_label_vote{VOTE_THRESHOLD}.csv", index=False)
    display(summary)


compare_models()